In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Load Hitters from public Rdatasets mirror
url = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/ISLR/Hitters.csv'
df = pd.read_csv(url)

# Normalize column name for Salary if needed
if 'salary' in df.columns and 'Salary' not in df.columns:
    df = df.rename(columns={'salary': 'Salary'})

# Drop rows with missing Salary
df = df.dropna(subset=['Salary']).copy()

# Drop index/unnamed column if present
for c in ['Unnamed: 0', 'X']:
    if c in df.columns:
        df = df.drop(columns=[c])

# Prepare features and target
y = df['Salary']
X = df.drop(columns=['Salary'])

# Convert categorical variables to dummies
X = pd.get_dummies(X, drop_first=True)

# Train/test split: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42)

# Number of predictors
p = X_train.shape[1]

# Fit RandomForestRegressor as a bagging-like model
rf = RandomForestRegressor(n_estimators=300, max_features=p, random_state=42)
rf.fit(X_train, y_train)

# Predict on test set and compute MSE
y_pred = rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

# Print test MSE rounded to 2 decimals
print(f"{mse:.2f}")


122578.32


In [5]:
# Feature importances DataFrame (sorted descending). Run this cell after fitting `rf` above.
import pandas as pd
rf2 = RandomForestRegressor(n_estimators=300, max_features=5, random_state=42)
rf2.fit(X_train, y_train)

# Ensure `rf` and `X_train` are available in the kernel. If not, run the previous cell that fits the model.
try:
    importances = rf2.feature_importances_
    feature_names = list(X_train.columns)
except NameError:
    raise NameError('`rf2` or `X_train` not found. Run the cell that fits the RandomForest first.')

fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)
fi_df = fi_df[~fi_df['feature'].str.contains("rownames_")]
# Show rounded importance for readability
fi_df['importance'] = fi_df['importance'].round(6)
print(fi_df)

top_var = fi_df.loc[0, 'feature']
top_val = fi_df.loc[0, 'importance']
print(f"\nTop variable by importance: {top_var} (importance = {top_val})")



        feature  importance
0        CWalks    0.089905
1         CHits    0.076918
2        CAtBat    0.073306
3         CRuns    0.068323
4          CRBI    0.066300
5        CHmRun    0.054732
6          Hits    0.051100
7           RBI    0.049241
8         AtBat    0.047732
9          Runs    0.046084
10        Walks    0.045810
11      PutOuts    0.039019
12        Years    0.038342
13        HmRun    0.037589
14       Errors    0.022309
15      Assists    0.021856
18   Division_W    0.009770
21     League_N    0.006015
23  NewLeague_N    0.005778

Top variable by importance: CWalks (importance = 0.089905)


In [7]:
# Decision tree on OJ dataset: fit and evaluate
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load OJ dataset
url = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/ISLR/OJ.csv'
oj = pd.read_csv(url)

# Drop possible index columns
for c in ['Unnamed: 0', 'X']:
    if c in oj.columns:
        oj = oj.drop(columns=[c])

# Ensure Purchase column is named correctly
if 'Purchase' not in oj.columns and 'purchase' in oj.columns:
    oj = oj.rename(columns={'purchase': 'Purchase'})

# Prepare predictors and target (use all variables except Purchase)
y = oj['Purchase']
X = oj.drop(columns=['Purchase'])
# Convert categorical variables to dummies
X = pd.get_dummies(X, drop_first=True)

# Train/test split: 75% training, 25% test
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, random_state=42)

# Fit Decision Tree with requested hyperparameters
dt = DecisionTreeClassifier(criterion='entropy', max_depth=4, min_samples_leaf=5, random_state=42)
dt.fit(X_train, y_train)

# Predict and evaluate
y_pred = dt.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy (rounded to 3 decimals): {acc:.3f}")

print('\nClassification report:')
print(classification_report(y_test, y_pred))

print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))

# Optional: show top feature importances
try:
    fi = pd.DataFrame({'feature': X.columns, 'importance': dt.feature_importances_})
    fi = fi.sort_values('importance', ascending=False).reset_index(drop=True)
    print('\nTop features by importance:')
    print(fi.head(10))
except Exception:
    pass


Test accuracy (rounded to 3 decimals): 0.817

Classification report:
              precision    recall  f1-score   support

          CH       0.79      0.94      0.86       159
          MM       0.88      0.64      0.74       109

    accuracy                           0.82       268
   macro avg       0.83      0.79      0.80       268
weighted avg       0.83      0.82      0.81       268


Confusion matrix:
[[149  10]
 [ 39  70]]

Top features by importance:
          feature  importance
0         LoyalCH    0.758735
1       PriceDiff    0.123779
2        rownames    0.034298
3   ListPriceDiff    0.026682
4       SpecialCH    0.025366
5     SalePriceMM    0.018697
6  WeekofPurchase    0.012442
7          DiscCH    0.000000
8         StoreID    0.000000
9         PriceCH    0.000000


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Load Auto dataset from GitHub
url = 'https://raw.githubusercontent.com/fedscornell/MachineLearning25/main/Auto.csv'
auto = pd.read_csv(url)

# Drop rows with missing values
auto = auto.dropna().copy()

# Create binary mpg_high: 1 if mpg > median, else 0
med = auto['mpg'].median()
auto['mpg_high'] = (auto['mpg'] > med).astype(int)

# Prepare X and y: exclude mpg and name
if 'name' in auto.columns:
    X = auto.drop(columns=['mpg','name','mpg_high'])
else:
    X = auto.drop(columns=['mpg','mpg_high'])

y = auto['mpg_high']

# Convert categorical variables to dummies (if any)
X = pd.get_dummies(X, drop_first=True)

# Train/test split 70/30
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42)

# Fit GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)

# Predict and compute accuracy
y_pred = gb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"{acc:.3f}")


0.890


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Load Hitters dataset
url = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/ISLR/Hitters.csv'
df = pd.read_csv(url)
# Drop index column if present
for c in ['Unnamed: 0', 'X']:
    if c in df.columns:
        df = df.drop(columns=[c])
# Drop rows with missing values
df = df.dropna().copy()
# Normalize Salary column
if 'Salary' not in df.columns and 'salary' in df.columns:
    df = df.rename(columns={'salary': 'Salary'})

# Prepare X and y
y = df['Salary']
X = df.drop(columns=['Salary'])
# Dummify categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train/test split 70/30
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42)

# Fit maximum-depth tree
base_tree = DecisionTreeRegressor(max_depth=6, random_state=42)
base_tree.fit(X_train, y_train)

# Cost complexity pruning path
path = base_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
# Remove duplicate/very small alphas and ensure array sorted
ccp_alphas = np.unique(ccp_alphas)

# Grid search over ccp_alpha with 5-fold CV
param_grid = {'ccp_alpha': ccp_alphas}
grid = GridSearchCV(DecisionTreeRegressor(max_depth=6, random_state=42), param_grid, cv=5,
                    scoring='neg_mean_squared_error', n_jobs=-1)
grid.fit(X_train, y_train)

best = grid.best_estimator_
# Number of leaf nodes in best estimator
try:
    n_leaves = best.get_n_leaves()
except Exception:
    # fallback: count children == -1
    tree = best.tree_
    n_leaves = np.sum(tree.children_left == -1)

# Compute test MSE of the pruned (best) tree
y_pred_test = best.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred_test)

# Print results
print(f"n_leaves: {n_leaves}")
print(f"test_mse: {mse_test:.2f}")


n_leaves: 4
test_mse: 161392.55
